In [ ]:
import sys; sys.path.append('..')
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm
import MeshFEM, parallelism, benchmark, utils
import numpy.linalg as la

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

size = 10

n_tubes = 4
step = 20
n = n_tubes * step + 1  # Grid size

# Step 1: Generate grid points
x = np.linspace(0, size, n)
y = np.linspace(0, size, n)
X, Y = np.meshgrid(x, y)
Y += np.cos(X * 5) * 0.1
X += np.cos(Y * 5) * 0.1
Z = np.zeros_like(X)
vertices = np.vstack([X.ravel(), Y.ravel(), Z.ravel()]).T

# Step 2: Triangulate the grid
triangles = []
for i in range(n - 1):
    for j in range(n - 1):
        idx = i * n + j
        triangles.append([idx, idx + 1, idx + n])
        triangles.append([idx + 1, idx + n + 1, idx + n])

# Step 3: Create a Triangulation object for visualization
triangulation = mtri.Triangulation(vertices[:, 0], vertices[:, 1], triangles)

# Step 4: Visualize the mesh
plt.triplot(triangulation, 'o-')

fused_points_x = []
fused_points_y = []
for i in range(n):
    for j in range(n):
        if i == 0 or i == n - 1 or j == 0 or j == n - 1 or i % step == 0 or j % step == 0 or (j - step / 2) % step == 0:
            fused_points_x.append(X[i, j])
            fused_points_y.append(Y[i, j])
plt.scatter(fused_points_x, fused_points_y, c = 'black', zorder = 10)
plt.gca().set_aspect('equal')
plt.show()

In [ ]:
m = MeshFEM.Mesh(np.array(vertices), np.array(triangles))

In [ ]:
num_sheets = 4

In [ ]:
m.numVertices()

In [ ]:
reducedVarIdxForVertexOnSheet = np.zeros((m.numVertices(), num_sheets), dtype=int)

total_reduced_vars = 0

for i in range(n):
    for j in range(n):
        reducedVarIdxForVertexOnSheet[i * n + j, 0] = i * n + j

total_reduced_vars = n * n

for s in range(num_sheets - 2):
    for i in range(n):
        for j in range(n):
            if i == 0 or i == n - 1 or j == 0 or j == n - 1:
                reducedVarIdxForVertexOnSheet[i * n + j, s + 1] = reducedVarIdxForVertexOnSheet[i * n + j, s]
            elif i % step == 0:
                reducedVarIdxForVertexOnSheet[i * n + j, s + 1] = reducedVarIdxForVertexOnSheet[i * n + j, s]
            else:
                reducedVarIdxForVertexOnSheet[i * n + j, s + 1] = total_reduced_vars
                total_reduced_vars += 1

    for i in range(n):
        for j in range(n):
            if i == 0 or i == n - 1 or j == 0 or j == n - 1:
                reducedVarIdxForVertexOnSheet[i * n + j, s + 2] = reducedVarIdxForVertexOnSheet[i * n + j, s]
            if j % step == 0:
                reducedVarIdxForVertexOnSheet[i * n + j, s + 2] = reducedVarIdxForVertexOnSheet[i * n + j, s + 1]
            else:
                reducedVarIdxForVertexOnSheet[i * n + j, s + 2] = total_reduced_vars
                total_reduced_vars += 1



In [ ]:
reducedVarIdxForVertexOnSheet = np.zeros((m.numVertices(), num_sheets), dtype=int)

total_reduced_vars = 0

for i in range(n):
    for j in range(n):
        reducedVarIdxForVertexOnSheet[i * n + j, 0] = i * n + j

total_reduced_vars = n * n

s = 0
for i in range(n):
    for j in range(n):
        if i == 0 or i == n - 1 or j == 0 or j == n - 1:
            reducedVarIdxForVertexOnSheet[i * n + j, s + 1] = reducedVarIdxForVertexOnSheet[i * n + j, s]
        elif i % step == 0:
            reducedVarIdxForVertexOnSheet[i * n + j, s + 1] = reducedVarIdxForVertexOnSheet[i * n + j, s]
        else:
            reducedVarIdxForVertexOnSheet[i * n + j, s + 1] = total_reduced_vars
            total_reduced_vars += 1

for i in range(n):
    for j in range(n):
        if (j - step / 2) % step == 0 and (i - step / 2) % step == 0:
            reducedVarIdxForVertexOnSheet[i * n + j, s + 2] = reducedVarIdxForVertexOnSheet[i * n + j, s + 1]
        else:
            reducedVarIdxForVertexOnSheet[i * n + j, s + 2] = total_reduced_vars
            total_reduced_vars += 1

for i in range(n):
    for j in range(n):
        if i == 0 or i == n - 1 or j == 0 or j == n - 1:
            reducedVarIdxForVertexOnSheet[i * n + j, s + 3] = reducedVarIdxForVertexOnSheet[i * n + j, s + 2]
        elif j % step == 0:
            reducedVarIdxForVertexOnSheet[i * n + j, s + 3] = reducedVarIdxForVertexOnSheet[i * n + j, s + 2]
        else:
            reducedVarIdxForVertexOnSheet[i * n + j, s + 3] = total_reduced_vars
            total_reduced_vars += 1


In [ ]:
np.max(reducedVarIdxForVertexOnSheet)

In [ ]:
pressures = 0.025 * np.ones(num_sheets - 1)

In [ ]:
pressures[1] = 0

In [ ]:
misheet = inflation.MultilayerInflatable(m, num_sheets = num_sheets, pressures = pressures, reducedVarIdxForVertexOnSheet = reducedVarIdxForVertexOnSheet)
                               

In [ ]:
import py_newton_optimizer

In [ ]:

opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10


misheet.setUseTensionFieldEnergy(True)
misheet.setUseHessianProjectedEnergy(False)

In [ ]:
from visualization import TriMeshViewer

In [ ]:
misheet.visualizationMesh()

In [ ]:
# First generate results with fixed boundary 
viewer = TriMeshViewer(misheet, width=768, height=640)
viewer.showWireframe(False)

# viewer.setCameraParams(((1.613494603240345, -3.9332708615926393, 1.4922998234349831),
# (-0.05948468564942635, 0.33267929672385665, 0.941162078339598),
# (0.0, 0.0, 0.0)))

viewer.update(scalarField=utils.getStrains(misheet)[:, 0])   
viewer.show()

In [ ]:
def cb(it):
    # viewer.update(scalarField=utils.getStrains(misheet)[:, 0])   
    viewer.update()


In [ ]:
# import fd_validation

# misheet.setVars(misheet.getVars() + np.random.random(misheet.numVars()))

# viewer.update()

# fd_validation.gradConvergencePlot(misheet, customArgs = {"energyType": inflation.MultilayerInflatable.EnergyType.Pressure})

# fd_validation.hessConvergencePlot(misheet, customArgs = {"energyType": inflation.MultilayerInflatable.EnergyType.Elastic})

# fd_validation.hessConvergencePlot(misheet, customArgs = {"energyType": inflation.MultilayerInflatable.EnergyType.Pressure})

In [ ]:
misheet.pressure

In [ ]:
misheet.pressure = [0.00, 0.0, 0.055]

In [ ]:

fixedVars = []
hessian_shift = 1e-6

opts.niter = 100
opts.gradTol = 1e-5

benchmark.reset()
cr = inflation.inflation_newton(misheet, fixedVars, opts, hessianShift = hessian_shift, callback = cb)
benchmark.report()


In [ ]:
misheet.hessian()